# Import and Setup

In [1]:
import pandas as pd

# Load Data

In [24]:
data_path = '../data/processed/processed_data.csv'
event_path = '../data/processed/event.csv'

df = pd.read_csv(data_path)
event_df = pd.read_csv(event_path)

# Normalize Time

In [ ]:
def timestamp_to_seconds(time: str) -> int:
    """Convert a timestamp in 'HH:MM:SS' format to total seconds.

    Args:
        time (str): Timestamp in 'HH:MM:SS' format.

    Returns:
        int: Total seconds.
    """
    h, m, s = map(int, time.split(':'))
    return h * 3600 + m * 60 + s

def normalize_time(df: pd.DataFrame,
                   col_name: str, 
                   base_time: int) -> pd.DataFrame:
    """Normalize the time column in the DataFrame by subtracting the base time.

    Args:
        df (pd.DataFrame): DataFrame containing the time column.
        col_name (str): Name of the column to normalize.
        base_time (int): Base time in seconds to subtract.
    
    Returns:
        pd.DataFrame: DataFrame with normalized time column.
    """
    df[col_name] = df[col_name].apply(lambda x: timestamp_to_seconds(x) - base_time)
    return df

In [25]:
base_time = timestamp_to_seconds(df['occurence_timestamp'].iloc[0])
print(f"Base time (in seconds): {base_time}")

print("Normalizing 'occurence_timestamp' in df\n")
normalized_df = normalize_time(df, 'occurence_timestamp', base_time)
normalized_df.head()

Base time (in seconds): 12330
Normalizing 'occurence_timestamp' in df



,message_type,occurence_timestamp,timeStampUTC,author,content,message,emoji,text_content
0,Chat Message,0,2026-01-24 10:50:55.099409,@yamin12345-z,"{'message': '#7 ', 'emoji': [], 'text_content'...",#7,[],#7
1,Chat Message,0,2026-01-24 10:50:55.222274,@fdoniaquilani9902,"{'message': 'Suport Indo ', 'emoji': [], 'text...",Suport Indo,[],Suport Indo
2,Chat Message,0,2026-01-24 10:50:55.256967,@theycalledmeamadman2772,"{'message': 'ONIC MANA BISA ', 'emoji': [], 't...",ONIC MANA BISA,[],ONIC MANA BISA
3,Chat Message,0,2026-01-24 10:50:55.363708,@loveydumpies,"{'message': 'AE plss ', 'emoji': [], 'text_con...",AE plss,[],AE plss
4,Chat Message,0,2026-01-24 10:50:55.592918,@Amrik199,"{'message': 'hina AE biar menang terus ', 'emo...",hina AE biar menang terus,[],hina AE biar menang terus


In [26]:
print("Normalizing 'timestamp' in event_df\n")
normalized_event_df = normalize_time(event_df, 'timestamp', base_time)
normalized_event_df.head()

Normalizing 'timestamp' in event_df



,event_id,timestamp,event,team,detail
0,E01,0,draft_dimulai,NaN,Fase draft Game 5 dimulai
1,E02,47,draft_pick,Alter Ego,Alter Ego memilih hero Uranus
2,E03,237,draft_unpredictable,Alter Ego,Uranus digunakan sebagai jungler dan draft sel...
3,E04,404,turtle_spawn_1,NaN,Turtle pertama muncul
4,E05,413,first_blood,Team Liquid,Team Liquid mendapatkan first blood sekaligus ...


In [28]:
normalized_df = normalized_df[['occurence_timestamp', 'text_content']]

# Event Categorization

In [29]:
EVENT_CATEGORIES = {
    'major': {
        'events': ['war_besar_lord', 'war_besar_turtle', 'war_besar_base', 
                   'match_selesai', 'perubahan_momentum'],
        'window_seconds': 90
    },
    'medium': {
        'events': ['first_blood', 'first_turret', 'lord_diamankan', 
                   'defend_lord', 'draft_unpredictable'],
        'window_seconds': 60
    },
    'minor': {
        'events': ['draft_dimulai', 'draft_pick', 'turtle_spawn_1', 
                   'turtle_spawn_3', 'lord_spawn_1', 'lord_spawn_2',
                   'kill', 'skirmish_trade', 'pickoff', 'highlight_replay'],
        'window_seconds': 30
    }
}

In [30]:
def get_event_category(event_name: str) -> str:
    """Get the category (major/medium/minor) for an event.
    
    Args:
        event_name: The event type name
    
    Returns:
        Category string: 'major', 'medium', or 'minor'
    """
    for category, config in EVENT_CATEGORIES.items():
        if event_name in config['events']:
            return category
    return 'minor'  # Default to minor for unknown events


def get_window_seconds(event_name: str) -> int:
    """Get the window size in seconds for an event.
    
    Args:
        event_name: The event type name
    
    Returns:
        Window size in seconds
    """
    category = get_event_category(event_name)
    return EVENT_CATEGORIES[category]['window_seconds']

In [31]:
# Apply categorization to event dataframe
normalized_event_df['category'] = normalized_event_df['event'].apply(get_event_category)
normalized_event_df['window_seconds'] = normalized_event_df['event'].apply(get_window_seconds)

# Calculate window boundaries (start and end times)
normalized_event_df['window_start'] = normalized_event_df['timestamp'] - normalized_event_df['window_seconds']
normalized_event_df['window_end'] = normalized_event_df['timestamp'] + normalized_event_df['window_seconds']

# Ensure window_start doesn't go below 0
normalized_event_df['window_start'] = normalized_event_df['window_start'].clip(lower=0)

normalized_event_df[['event_id', 'timestamp', 'event', 'category', 'window_seconds', 'window_start', 'window_end']]

,event_id,timestamp,event,category,window_seconds,window_start,window_end
0,E01,0,draft_dimulai,minor,30,0,30
1,E02,47,draft_pick,minor,30,17,77
2,E03,237,draft_unpredictable,medium,60,177,297
3,E04,404,turtle_spawn_1,minor,30,374,434
4,E05,413,first_blood,medium,60,353,473
5,E06,623,first_turret,medium,60,563,683
6,E07,640,kill,minor,30,610,670
7,E08,649,skirmish_trade,minor,30,619,679
8,E09,675,turtle_spawn_3,minor,30,645,705
9,E10,713,war_besar_turtle,major,90,623,803


In [32]:
# Summary of event categories
print("Event Category Summary:")
print("=" * 50)
for category in ['major', 'medium', 'minor']:
    count = (normalized_event_df['category'] == category).sum()
    window = EVENT_CATEGORIES[category]['window_seconds']
    print(f"{category.upper():8} | {count:2} events | ±{window}s window")
print("=" * 50)
print(f"Total: {len(normalized_event_df)} events")

Event Category Summary:
MAJOR    |  6 events | ±90s window
MEDIUM   |  6 events | ±60s window
MINOR    | 10 events | ±30s window
Total: 22 events


# Extract Comments per Event Window

In [33]:
def extract_comments_in_window(comments_df: pd.DataFrame, 
                                window_start: int, 
                                window_end: int) -> pd.DataFrame:
    """Extract comments that fall within a time window.
    
    Args:
        comments_df: DataFrame with 'occurence_timestamp' column
        window_start: Start of window (in seconds)
        window_end: End of window (in seconds)
    
    Returns:
        Filtered DataFrame with comments in the window
    """
    mask = (comments_df['occurence_timestamp'] >= window_start) & \
           (comments_df['occurence_timestamp'] <= window_end)
    return comments_df[mask].copy()

In [34]:
# Preview: Count comments per event window
print("Comments per Event Window:")
print("=" * 70)
print(f"{'Event ID':<10} {'Event':<25} {'Category':<8} {'Comments':>10}")
print("-" * 70)

for _, event_row in normalized_event_df.iterrows():
    comments_in_window = extract_comments_in_window(
        normalized_df,
        event_row['window_start'],
        event_row['window_end']
    )
    print(f"{event_row['event_id']:<10} {event_row['event']:<25} {event_row['category']:<8} {len(comments_in_window):>10}")

print("=" * 70)

Comments per Event Window:
Event ID   Event                     Category   Comments
----------------------------------------------------------------------
E01        draft_dimulai             minor           279
E02        draft_pick                minor           579
E03        draft_unpredictable       medium         1451
E04        turtle_spawn_1            minor           502
E05        first_blood               medium         1142
E06        first_turret              medium         1315
E07        kill                      minor           672
E08        skirmish_trade            minor           661
E09        turtle_spawn_3            minor           665
E10        war_besar_turtle          major          1897
E11        lord_spawn_1              minor           693
E12        pickoff                   minor           548
E13        war_besar_lord            major          1468
E14        perubahan_momentum        major          1633
E15        defend_lord               medium    